In [ ]:
# --- INSTALACIÓN DE LIBRERÍAS ---
!pip install ultralytics kagglehub

import kagglehub
import os
from ultralytics import YOLO

# --- DESCARGA DEL DATASET ---
print(" Descargando dataset de Kaggle...")
dataset_path = kagglehub.dataset_download("nikolasgegenava/sard-2-search-and-rescue-dataset-extra-classes")

print(f" Dataset descargado en: {dataset_path}")

# --- BÚSQUEDA AUTOMÁTICA DEL YAML ---
yaml_path = None
for root, dirs, files in os.walk(dataset_path):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path:
    print(f" Archivo de configuración encontrado en: {yaml_path}")
else:
    print(" ERROR: No se encontró 'data.yaml' en el dataset descargado. Revisa la descarga.")

In [ ]:
# --- CONFIGURACIÓN DEL MODELO ---
model = YOLO('yolov8s.pt')

if yaml_path:
    print(" INICIANDO ENTRENAMIENTO...")

    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        name='yolov8_sard_rescue',
        patience=15,

        # === DATA AUGMENTATION  ===
        degrees=15.0,      # Rotar imágenes +/- 15 grados
        fliplr=0.5,        # Efecto espejo horizontal (50% probabilidad)
        mosaic=1.0,        # Mezclar 4 imágenes (ayuda mucho a detectar objetos pequeños)
        mixup=0.15,        # Superponer imágenes (reduce falsos positivos)
        hsv_h=0.015,       # Variar tono de color ligeramente
        hsv_s=0.7,         # Variar saturación (simular sol fuerte o nublado)
        hsv_v=0.4,         # Variar brillo
    )
else:
    print(" No podemos entrenar porque falta el archivo yaml.")

In [ ]:
import os
import shutil
from google.colab import drive
from google.colab import files

# 1. CONECTAR GOOGLE DRIVE
if not os.path.exists('/content/drive'):
    print(" Conectando a Google Drive...")
    drive.mount('/content/drive')

# 2. RUTA DE ORIGEN
source_best = '/content/runs/detect/yolov8_sard_rescue/weights/best.pt'

# 3. RUTA DE DESTINO
dest_folder = '/content/drive/MyDrive/TFG'
dest_file = os.path.join(dest_folder, 'best_v8_sard.pt')

# 4. CREAR LA CARPETA SI NO EXISTE
if not os.path.exists(dest_folder):
    print(f" Creando carpeta {dest_folder}...")
    os.makedirs(dest_folder, exist_ok=True)

# 5. COPIAR EL ARCHIVO
if os.path.exists(source_best):
    shutil.copy(source_best, dest_file)
    print(f" ¡ÉXITO! Modelo guardado en Drive: {dest_file}")

    # 6. OPCIONAL: Descargar al PC
    print("⬇ Preparando descarga directa a tu PC...")
    #files.download(source_best)
else:
    print(f" ERROR: No encuentro el archivo generado en: {source_best}")
    print("Revisa la carpeta 'runs/detect' en el menú de la izquierda para ver el nombre real.")